In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("..") / "data" / "raw"
PROC_DIR = Path("..") / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Load the CSV you placed manually
df = pd.read_csv(RAW_DIR / "owid-covid-data.csv")

# Convert date
df["date"] = pd.to_datetime(df["date"])

# Make sure iso_code is string and filter out aggregates (OWID_* rows)
df["iso_code"] = df["iso_code"].astype(str)
mask_owid = df["iso_code"].str.startswith("OWID")          # True for OWID_* rows
df = df[mask_owid == False].copy()                         # keep only non-OWID

# Fill missing for new daily metrics
for col in ["new_cases", "new_deaths"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Forward-fill cumulative metrics by country
cum_cols = ["total_cases", "total_deaths", "people_vaccinated", "people_fully_vaccinated"]
cum_cols = [c for c in cum_cols if c in df.columns]        # keep only existing columns

df = df.sort_values(["location", "date"])
df[cum_cols] = df.groupby("location")[cum_cols].ffill()

clean_path = PROC_DIR / "covid_clean.csv"
df.to_csv(clean_path, index=False)
clean_path

PosixPath('../data/processed/covid_clean.csv')